# Plot rasterplot with the trial-averaged traces per animal.

It uses the `features.csv` file generated by `features-from-dlc`.

What this notebook does:

1. Sorts the data by `condition` or `animal`.
2. Calculates the average of the feature for each animal, condition and time.
3. Plots the raster for each feature.

How to use the notebook:

1. Replace with the full path to your `features.csv` file.
2. Replace with the full path to the directory where figures will be saved.
3. List the exact column (feature) names from the csv file you want to plot.
4. Map each feature name to its display label on the colorbar.
5. Choose how to sort the rows of the raster plot (animal/condition) and provide the the list.
6. Run All

In [11]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
import os

In [ ]:
# Parameters
# full path to the features.csv file (1)
filepath = r"D:\path\to\features.csv"
# path to the save directory (2)
output_dir = r"D:\path\to\save\directory"

# exact name of the features to plot, as they appear in the csv file (3)
features = ["speed", "theta_body", "theta_neck"]

# map feature name to display name (4)
clabels = {
    "speed": "speed (cm/s)",
    "theta_body": "body angle (°)",
    "theta_neck": "neck angle (°)",
}

# whether to sort by "condition" or by "animal"
# "animal" can be used to order by rostro-caudal injection level
# (conditions will always be splitted with a red line) (5)
sortby = "animal"
# order (list of animal ids if sort by animals; list of conditions if sort by conditions)
sortby_order = ["animal1", "animal2"]

# colormap : choose from https://matplotlib.org/stable/users/explain/colors/colormaps.html#sequential
cmap = "viridis"
# colormap : set highest value to this quantile in the data (remove outliers)
quantile = 0.99

In [ ]:
# Load file and display data
df_ffd = pd.read_csv(filepath)
display(df_ffd.head())

In [ ]:
# Group by animal, condition and time to extract individual time series,
# and average them for each animal, keeping the animal ID and condition information
df = df_ffd.groupby(["animal", "time", "condition"])[features].mean().reset_index()
# sort by animal order
df = df.set_index(sortby).loc[sortby_order].reset_index()
display(df.head())

In [5]:
# get mapping between animals and condition
map_animal_cond = (
    df[["animal", "condition"]].drop_duplicates(ignore_index=True).set_index("animal")
)
map_animal_cond = map_animal_cond.to_dict()["condition"]

In [6]:
# get delimiters location index
ser = (
    df.groupby(["animal"])["condition"]
    .unique()
    .str[0]
    .reset_index()
    .set_index(sortby)
    .loc[sortby_order]
    .reset_index()["condition"]
)
ind_delimiter = np.where(ser.ne(ser.shift().bfill()))[0]

In [ ]:
%matplotlib inline
# get shapes
nfeatures = len(features)
time = df["time"].unique()
ntimes = len(time)
animals = df["animal"].unique()
nanimals = len(animals)

for feature in features:
    # get values range
    crange = [df[feature].quantile(1 - quantile), df[feature].quantile(quantile)]

    # prepare figure
    fig, ax = plt.subplots(figsize=(8, 6))
    fig.subplots_adjust(right=0.9)

    # prepare data
    vals = df[feature].values
    expected_size = nanimals * ntimes
    actual_size = len(vals)

    if actual_size < expected_size:
        warnings.warn(f"Insufficient data : {actual_size} < {expected_size}. Truncates the dimensions.")
        ntimes = actual_size // nanimals
        vals = vals[:nanimals * ntimes]
    elif actual_size > expected_size:
        warnings.warn(f"Too much data : {actual_size} > {expected_size}. Trim the excess.")
        vals = vals[:expected_size]

    data = np.reshape(vals, (nanimals, ntimes))


    # plot raster
    p = ax.pcolormesh(
        time,
        animals,
        data,
        vmin=crange[0],
        vmax=crange[1],
    )

    # plot delimiters between conditions
    _ = [ax.axhline(loc - 0.5, linewidth=1.5, color="#ec1b8a") for loc in ind_delimiter]

    # flip plot so that first row is first condition
    ax.invert_yaxis()

    # add colorbar
    plt.colorbar(p, label=clabels[feature])

    # change y axis labels to show conditions
    ylabels = [
        f"{item.get_text()}-{map_animal_cond[item.get_text()]}"
        for item in ax.get_yticklabels()
    ]
    ax.set_yticks(animals)
    ax.set_yticklabels(ylabels)

    # xlabel
    ax.set_xlabel("time (s)")

    # stim
    ax.axvline(0, color="#1b8aec")

    # Save plot
    fig.savefig(os.path.join(output_dir, f"Mean_raster_plot_{feature}.svg"), dpi=300)
    plt.close(fig)